# T2ICount — IDCIA qualitative analysis

Notebook này dùng để **phân tích kết quả inference đã lưu**, không thay thế `test.py`.

Mục tiêu:
- So sánh `raw` vs `autocontrast`.
- So sánh prompt generic `"cell"` vs staining-specific prompt.
- Tự chọn một số case đáng xem: collapse, prompt giúp/hại, autocontrast giúp/hại, false positive khi `GT=0`.
- Chỉ re-inference các case đã chọn để lấy **density map**, tránh chạy lại toàn bộ IDCIA.

> Baseline chính vẫn là raw IDCIA. Autocontrast chỉ là diagnostic ablation.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display

# Works when launched from either the repository root or notebooks/.
candidates = (Path.cwd(), Path.cwd().parent)
ROOT = next((path for path in candidates if (path / 'test.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook from T2ICount/ or T2ICount/notebooks/.')

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.paths import AssetPaths

assets = AssetPaths.from_sources()
RESULTS_DIR = ROOT / 'results'
RAW_CSV = RESULTS_DIR / 'idcia_predictions.csv'
AUTO_CSV = RESULTS_DIR / 'idcia_autocontrast_predictions.csv'

print('Repo root:', ROOT)
print('Asset root:', assets.root)
print('Raw CSV:', RAW_CSV)
print('Autocontrast CSV:', AUTO_CSV)


## 1. Đọc kết quả đã inference

Phần này **không load model**, nên chạy rất nhanh. Nếu bạn đặt tên CSV khác, chỉ cần sửa `RAW_CSV` và `AUTO_CSV` ở cell trên.


In [ ]:
raw = pd.read_csv(RAW_CSV)
auto = pd.read_csv(AUTO_CSV)

required = {
    "image", "staining", "gt",
    "pred_generic", "abs_error_generic",
    "pred_specific", "abs_error_specific",
}
for name, df in [("raw", raw), ("autocontrast", auto)]:
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{name} CSV thiếu columns: {sorted(missing)}")

print("Raw:", raw.shape)
print("Autocontrast:", auto.shape)

display(raw.head())


## 2. Ghép raw và autocontrast để phân tích

Các cột `*_gain_*` được định nghĩa sao cho **giá trị dương = phương án sau tốt hơn**.

- `prompt_gain_raw > 0`: specific prompt giảm absolute error so với `"cell"` trên raw image.
- `prompt_gain_auto > 0`: specific prompt giảm absolute error trên autocontrast.
- `autocontrast_gain_specific > 0`: autocontrast giúp specific prompt.
- `autocontrast_gain_generic > 0`: autocontrast giúp generic prompt.


In [ ]:
keep = [
    "image", "staining", "gt",
    "pred_generic", "abs_error_generic",
    "pred_specific", "abs_error_specific",
]

merged = raw[keep].merge(
    auto[keep],
    on=["image", "staining", "gt"],
    suffixes=("_raw", "_auto"),
    validate="one_to_one",
)

merged["prompt_gain_raw"] = (
    merged["abs_error_generic_raw"] - merged["abs_error_specific_raw"]
)
merged["prompt_gain_auto"] = (
    merged["abs_error_generic_auto"] - merged["abs_error_specific_auto"]
)
merged["autocontrast_gain_specific"] = (
    merged["abs_error_specific_raw"] - merged["abs_error_specific_auto"]
)
merged["autocontrast_gain_generic"] = (
    merged["abs_error_generic_raw"] - merged["abs_error_generic_auto"]
)

display(merged.head())


## 3. Tóm tắt per-staining

Bảng này giúp xem prompt/contrast có cải thiện nhất quán theo staining hay chỉ do vài outlier.


In [ ]:
def staining_summary(df, suffix):
    rows = []
    for staining, g in df.groupby("staining", sort=False):
        rows.append({
            "staining": staining,
            "N": len(g),
            "mean_gt": g["gt"].mean(),
            f"generic_mae_{suffix}": g[f"abs_error_generic_{suffix}"].mean(),
            f"specific_mae_{suffix}": g[f"abs_error_specific_{suffix}"].mean(),
        })
    return pd.DataFrame(rows)

summary_raw = staining_summary(merged, "raw")
summary_auto = staining_summary(merged, "auto")

summary = summary_raw.merge(
    summary_auto,
    on=["staining", "N", "mean_gt"],
    how="outer",
)

summary["specific_change_raw_to_auto"] = (
    summary["specific_mae_raw"] - summary["specific_mae_auto"]
)

display(summary.round(3))


## 4. Tự chọn các case đáng visualize

Notebook cố gắng chọn các ảnh khác nhau cho các failure/success mode:

1. **Raw collapse**: GT lớn nhưng generic prediction gần 0.
2. **Specific prompt helps**: specific prompt giảm error nhiều nhất trên raw.
3. **Specific prompt hurts**: specific prompt làm error tăng nhiều nhất trên raw.
4. **Autocontrast helps**: autocontrast giảm specific-prompt error nhiều nhất.
5. **Autocontrast hurts**: autocontrast làm specific-prompt error tăng nhiều nhất.
6. **Zero-GT false positive**: ảnh `GT=0` nhưng specific/autocontrast prediction lớn.

Đây chỉ là auto-selection để tìm case nhanh. Bạn có thể thay `selected_images` bằng danh sách ảnh thủ công sau khi xem CSV.


In [ ]:
def pick_unique(candidates, used):
    for _, row in candidates.iterrows():
        if row["image"] not in used:
            used.add(row["image"])
            return row
    return None

used = set()
picked = []

collapse = merged[
    (merged["gt"] > 0) & (merged["pred_generic_raw"] < 1.0)
].sort_values("gt", ascending=False)
row = pick_unique(collapse, used)
if row is not None:
    picked.append(("raw_generic_collapse", row))

row = pick_unique(merged.sort_values("prompt_gain_raw", ascending=False), used)
if row is not None:
    picked.append(("specific_prompt_helps_raw", row))

row = pick_unique(merged.sort_values("prompt_gain_raw", ascending=True), used)
if row is not None:
    picked.append(("specific_prompt_hurts_raw", row))

row = pick_unique(
    merged.sort_values("autocontrast_gain_specific", ascending=False), used
)
if row is not None:
    picked.append(("autocontrast_helps_specific", row))

row = pick_unique(
    merged.sort_values("autocontrast_gain_specific", ascending=True), used
)
if row is not None:
    picked.append(("autocontrast_hurts_specific", row))

zero_gt = merged[merged["gt"] == 0].copy()
if len(zero_gt):
    zero_gt["max_specific_pred"] = zero_gt[
        ["pred_specific_raw", "pred_specific_auto"]
    ].max(axis=1)
    row = pick_unique(
        zero_gt.sort_values("max_specific_pred", ascending=False), used
    )
    if row is not None:
        picked.append(("zero_gt_false_positive", row))

selected = pd.DataFrame([
    {
        "reason": reason,
        "image": row["image"],
        "staining": row["staining"],
        "gt": row["gt"],
        "pred_generic_raw": row["pred_generic_raw"],
        "pred_specific_raw": row["pred_specific_raw"],
        "pred_generic_auto": row["pred_generic_auto"],
        "pred_specific_auto": row["pred_specific_auto"],
        "prompt_gain_raw": row["prompt_gain_raw"],
        "autocontrast_gain_specific": row["autocontrast_gain_specific"],
    }
    for reason, row in picked
])

display(selected.round(3))

selected_images = selected["image"].tolist()


## 5. Load T2ICount để lấy density map cho các case đã chọn

Từ đây mới cần GPU/checkpoint. Notebook re-inference **chỉ các ảnh đã chọn**, không chạy lại 53 ảnh.

Nếu chỉ muốn phân tích CSV, bạn có thể dừng trước phần này.


In [ ]:
import torch

from datasets.dataset import IDCIA
from models.build import build_t2icount
from utils.inference import build_prompt_attention_mask, predict_density
from test import IDCIA_PROMPTS

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CROP_SIZE = 384
PATCH_BATCH_SIZE = 1
IDCIA_ROOT = assets.dataset_dir('idcia')

model = build_t2icount(
    ROOT / 'configs' / 'v1-inference.yaml',
    assets.sd_checkpoint,
    assets.clip_dir,
    checkpoint_path=assets.official_checkpoint,
    device=DEVICE,
    mode='eval',
)
tokenizer = model.clip.tokenizer

raw_ds = IDCIA(IDCIA_ROOT, split='test', preprocess='raw')
auto_ds = IDCIA(IDCIA_ROOT, split='test', preprocess='autocontrast')
raw_index = {sample['image_name']: i for i, sample in enumerate(raw_ds.samples)}
auto_index = {sample['image_name']: i for i, sample in enumerate(auto_ds.samples)}

print('Device:', DEVICE)
print('Model loaded from external assets.')


## 6. Helper: density-map inference

Logic này giữ nguyên inference math của `test.py`:
- `extract_patches`
- model forward
- `reassemble_patches`
- chia density map cho `60`

Khác `test.py` ở chỗ helper trả về **density map**, không chỉ tổng count.


In [ ]:
def predict_density_map(model, inputs, prompt, batch_size=1, crop_size=384):
    prompt_mask = build_prompt_attention_mask(tokenizer, prompt)
    result = predict_density(
        model,
        inputs,
        prompt,
        prompt_mask,
        batch_size=batch_size,
        patch_size=crop_size,
        stride=crop_size,
    )
    density_np = result.squeeze().detach().cpu().numpy()
    pred_count = float(result.sum().item())
    return density_np, pred_count


def get_case(image_name, preprocess):
    if preprocess == "raw":
        ds = raw_ds
        idx = raw_index[image_name]
    elif preprocess == "autocontrast":
        ds = auto_ds
        idx = auto_index[image_name]
    else:
        raise ValueError(preprocess)

    tensor, gt, _, staining = ds[idx]
    sample = ds.samples[idx]

    with Image.open(sample["image_path"]) as im:
        source = im.convert("L")
        if preprocess == "autocontrast":
            source = ImageOps.autocontrast(source)
        display_image = np.asarray(source)

    specific_prompt = IDCIA_PROMPTS[staining]
    inputs = tensor.unsqueeze(0).to(DEVICE)

    den_generic, pred_generic = predict_density_map(
        model, inputs, "cell",
        batch_size=PATCH_BATCH_SIZE,
        crop_size=CROP_SIZE,
    )
    den_specific, pred_specific = predict_density_map(
        model, inputs, specific_prompt,
        batch_size=PATCH_BATCH_SIZE,
        crop_size=CROP_SIZE,
    )

    return {
        "image": image_name,
        "staining": staining,
        "gt": int(gt),
        "preprocess": preprocess,
        "specific_prompt": specific_prompt,
        "display_image": display_image,
        "density_generic": den_generic,
        "density_specific": den_specific,
        "pred_generic": pred_generic,
        "pred_specific": pred_specific,
    }


## 7. Visualize một case

Mặc định hiển thị:
1. Input image.
2. Generic `"cell"` density map.
3. Specific-prompt density map.

Bạn có thể gọi cùng một `image_name` với `preprocess="raw"` và `"autocontrast"` để so sánh trực tiếp.


In [ ]:
def visualize_case(image_name, preprocess="raw"):
    case = get_case(image_name, preprocess)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].imshow(case["display_image"], cmap="gray")
    axes[0].set_title(
        f'{case["staining"]} | GT={case["gt"]}\n{case["preprocess"]}'
    )

    axes[1].imshow(case["density_generic"])
    axes[1].set_title(
        f'Prompt: "cell"\nPred={case["pred_generic"]:.2f}'
    )

    axes[2].imshow(case["density_specific"])
    axes[2].set_title(
        f'Prompt: "{case["specific_prompt"]}"\n'
        f'Pred={case["pred_specific"]:.2f}'
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    return case


# Ví dụ:
# visualize_case(selected_images[0], preprocess="raw")


## 8. So sánh raw vs autocontrast cho cùng một ảnh

Cell dưới re-inference cùng một ảnh hai lần. Đây là phần hữu ích nhất để xem autocontrast chỉ làm model active hơn hay thực sự đưa density về đúng vùng.


In [ ]:
def compare_raw_vs_autocontrast(image_name):
    print("RAW")
    raw_case = visualize_case(image_name, "raw")

    print("AUTOCONTRAST")
    auto_case = visualize_case(image_name, "autocontrast")

    comparison = pd.DataFrame([
        {
            "preprocess": "raw",
            "GT": raw_case["gt"],
            "generic_pred": raw_case["pred_generic"],
            "specific_pred": raw_case["pred_specific"],
        },
        {
            "preprocess": "autocontrast",
            "GT": auto_case["gt"],
            "generic_pred": auto_case["pred_generic"],
            "specific_pred": auto_case["pred_specific"],
        },
    ])
    display(comparison.round(3))
    return raw_case, auto_case


# Ví dụ:
# compare_raw_vs_autocontrast(selected_images[0])


## 9. Chạy các case đã chọn

Không nên chạy tất cả ngay nếu inference chậm. Xem `selected` trước rồi chọn 4–6 ảnh thực sự có ý nghĩa cho báo cáo.


In [ ]:
# for image_name in selected_images:
#     compare_raw_vs_autocontrast(image_name)


## Gợi ý chọn figure cho báo cáo

Sau khi xem density maps, ưu tiên giữ khoảng 4–6 case:

- Một ảnh raw mà cả hai prompt gần như không tạo density.
- Một ảnh specific prompt cải thiện rõ rệt.
- Một ảnh specific prompt gây false positive/overcount.
- Một ảnh autocontrast làm density hợp lý hơn.
- Một ảnh autocontrast làm activation mạnh nhưng sai vùng.
- Nếu có, một ảnh `GT=0` nhưng model tạo density lớn.

Không cần cố chọn case "đẹp"; mục tiêu của V0.5 là giải thích **model đang sai theo kiểu nào** trước khi thay loss/training.
